In [1]:
!nvidia-smi -L
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — set Runtime to GPU before continuing'

GPU 0: Tesla T4 (UUID: GPU-738b8532-3f0e-7187-a7a7-e0c85724396b)
CUDA available: True


In [2]:
%cd /content
# DAVI code lives on the feature/davi branch.
![ -d Rubic-RFL ] && (cd Rubic-RFL && git fetch origin && git checkout feature/davi && git pull) || git clone --branch feature/davi https://github.com/IricsDo/Rubic-RFL.git
%cd /content/Rubic-RFL

/content
Cloning into 'Rubic-RFL'...
remote: Enumerating objects: 567, done.
remote: Counting objects: 100% (567/567), done.
remote: Compressing objects: 100% (322/322), done.
remote: Total 567 (delta 254), reused 538 (delta 225), pack-reused 0 (from 0)
Receiving objects: 100% (567/567), 835.25 KiB | 2.91 MiB/s, done.
Resolving deltas: 100% (254/254), done.
/content/Rubic-RFL


In [3]:
!pip install -q -e backend   # provides app.cube  (required by the trainer)
!pip install -q -e rl        # provides rubic_rl

# Make both packages importable in THIS running kernel without a restart:
import sys
for p in ('/content/Rubic-RFL/backend', '/content/Rubic-RFL/rl'):
    if p not in sys.path:
        sys.path.insert(0, p)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 85.8 MB/s eta 0:00:00:00:010:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 118.8 MB/s eta 0:00:0000:01
  Building editable for rubic-rfl-backend (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rubic-rfl-rl (pyproject.toml) ... done


In [4]:
from app.cube import Cube
from rubic_rl.training import adi
print('cross-package imports OK')

cross-package imports OK


In [5]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/rubic-rfl'
import os
for sub in ('datasets', 'checkpoints', 'reports'):
    os.makedirs(f'{OUT}/{sub}', exist_ok=True)
print('Saving outputs under', OUT)

Mounted at /content/drive
Saving outputs under /content/drive/MyDrive/rubic-rfl


# -----

In [6]:
%cd /content/Rubic-RFL/rl

!python -m rubic_rl.training.davi \
  --device cuda \
  --iterations 100000 \
  --batch-size 1000 \
  --hidden-dim 512 \
  --residual-blocks 6 \
  --dropout 0.0 \
  --learning-rate 0.001 \
  --weight-decay 0.00001 \
  --loss-type smooth_l1 \
  --huber-delta 1.0 \
  --policy-loss-weight 0.25 \
  --grad-clip-norm 5.0 \
  --target-update-interval 200 \
  --checkpoint-interval 200 \
  --curriculum-start 1 \
  --curriculum-interval 150 \
  --max-scramble-depth 30 \
  --hard-depth-fraction 0.5 \
  --seed 20260603 \
  --model-version torch-value-davi-v0.3 \
  --checkpoint-out /content/drive/MyDrive/rubic-rfl/checkpoints/torch-value-davi.pt \
  --report-out /content/drive/MyDrive/rubic-rfl/reports/davi-training.json

/content/Rubic-RFL/rl
[davi] resumed from /content/drive/MyDrive/rubic-rfl/checkpoints/torch-value-davi.pt.resume at step 80000
[davi] step 80001/100000 loss=0.8125 K=30 depth_mean=22.7 y_mean=10.23 pred_mean=10.25 mae=0.271 pi_acc=0.050 1.2s
[davi] step 80050/100000 loss=0.7847 K=30 depth_mean=22.6 y_mean=10.20 pred_mean=10.22 mae=0.297 pi_acc=0.148 4.9s
[davi] step 80100/100000 loss=0.7559 K=30 depth_mean=22.7 y_mean=10.26 pred_mean=10.03 mae=0.388 pi_acc=0.186 8.6s
[davi] step 80150/100000 loss=0.6833 K=30 depth_mean=22.7 y_mean=10.15 pred_mean=10.17 mae=0.317 pi_acc=0.231 13.2s
[davi] step 80200/100000 loss=0.6636 K=30 depth_mean=23.0 y_mean=10.39 pred_mean=10.35 mae=0.318 pi_acc=0.254 16.8s
[davi] step 80250/100000 loss=0.6449 K=30 depth_mean=22.7 y_mean=10.26 pred_mean=10.24 mae=0.299 pi_acc=0.253 22.2s
[davi] step 80300/100000 loss=0.6505 K=30 depth_mean=22.9 y_mean=10.26 pred_mean=10.41 mae=0.310 pi_acc=0.255 26.8s
[davi] step 80350/100000 loss=0.6440 K=30 depth_mean=23.0 y_mea

In [1]:
%cd /content/Rubic-RFL/rl

!python -m rubic_rl.evaluation.davi_eval \
  --model /content/drive/MyDrive/rubic-rfl/checkpoints/torch-value-davi.pt \
  --depths 15 \
  --samples-per-depth 50 \
  --weight 0.6 \
  --policy-weight 0.25 \
  --batch-expansion 1000 \
  --max-nodes 1000000 \
  --device cuda \
  --out /content/drive/MyDrive/rubic-rfl/reports/davi-eval-depth-15-policy-v03-s50.json

/content/Rubic-RFL/rl


: 